# Explore EDD: golden set + diagnose

Hands-on run through the middle of the loop: **trace → eval → diagnose**.

1. Look at the golden set (44 cases)
2. Run a single case — answer / citations / refusal
3. Categorise a batch (diagnose)
4. **Key experiment**: re-run the `wrongly_refused` cases through **multi-hop** (`answer_agentic`) and measure recovery
5. Inspect a trace

Tweak anything, add cells freely. Each answerable case = one Claude call (a few cents).

In [ ]:
import os
from pathlib import Path

# Make cwd = repo root so data/ and evals/ paths resolve
while not Path("pyproject.toml").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
print("cwd:", Path.cwd())

import json
import pandas as pd

from policy_copilot.agent import answer
from policy_copilot.tool_agent import answer_agentic
from policy_copilot.index import load_index

golden = [json.loads(l) for l in Path("evals/golden.jsonl").read_text().splitlines() if l.strip()]
print(len(golden), "golden cases")

# Load the index once (the embedding model takes ~10s to warm up)
index, chunks = load_index()
print("index ready")

## 1. What the golden set looks like

44 cases: 40 answerable (from FinanceBench, with gold answer + evidence) + 4 hand-authored refusals.

In [ ]:
g = pd.DataFrame(golden)
print("types:", g["type"].value_counts().to_dict())
print("by doc:", g[g.type == "answerable"]["doc_name"].value_counts().to_dict())
g[["id", "type", "doc_name", "question"]].head(12)

## 2. Run a single case (change `idx` to try different ones)

The first cases are mostly **complex analysis questions** (need synthesis/calculation) — good for seeing how strict grounding reacts.

In [ ]:
idx = 0  # <- change this index to try a different question
case = golden[idx]
print("Q   :", case["question"])
print("gold:", case.get("gold_answer", "(refusal case)")[:200])

res = answer(case["question"], index, chunks)
print("\nrefused  :", res.refused)
print("answer   :", res.text[:300])
print("citations:", res.citations)

## 3. Diagnose: categorise a batch

This is the core of `evals/run_eval.py` — run the set, bin each outcome, look at the failure modes. Raise `N` for more coverage (slower).

In [ ]:
def classify(case, res):
    if case["type"] == "refusal":
        return "ok" if res.refused else "should_have_refused"
    if res.refused:
        return "wrongly_refused"
    exp = case.get("expected_contains") or []
    if exp and not any(e in res.text for e in exp):
        return "missing_expected"
    if not res.citations:
        return "no_citation"
    return "ok"


N = 8  # <- how many answerable cases to run (one Claude call each)
sample = [c for c in golden if c["type"] == "answerable"][:N]
sample += [c for c in golden if c["type"] == "refusal"]

rows = []
for c in sample:
    r = answer(c["question"], index, chunks)
    rows.append({"id": c["id"], "type": c["type"], "cat": classify(c, r), "q": c["question"][:50]})
diag = pd.DataFrame(rows)
print(diag["cat"].value_counts().to_dict())
diag

## 4. ⭐ Key experiment: re-run `wrongly_refused` through multi-hop

**Hypothesis**: these were refused because single-hop + strict grounding can't handle multi-step analysis.

**Test**: run the *same* questions through `answer_agentic` (Claude decides when/what to search, multiple times) and check whether `refused` flips to False with citations that pass the verifier. This closes the **diagnose → fix → re-measure** loop.

Control the variables: same questions, only single-hop → multi-hop changes.

In [ ]:
failed = diag[diag.cat == "wrongly_refused"]["id"].tolist()
print("wrongly_refused:", failed)

comp = []
for cid in failed:
    c = next(x for x in golden if x["id"] == cid)  # look the full case back up by id
    a = answer_agentic(c["question"], index, chunks)  # same question, now multi-hop
    comp.append({
        "id": cid,
        "q": c["question"][:45],
        "agentic_refused": a.refused,  # still refused? False = recovered
        "n_search": len(a.tool_calls),  # how many times it searched (multi-hop in action)
        "verified": a.verdict.citations_resolve and a.verdict.numbers_verbatim,
        "answer": a.text[:100],
    })
comp_df = pd.DataFrame(comp)
recovered = 0 if comp_df.empty else int((~comp_df["agentic_refused"]).sum())
print(f"recovered (no longer refused): {recovered} / {len(failed)}")
comp_df

## 5. Inspect a trace

Every answer is written to `data/traces/traces.jsonl`; if Langfuse is running, see the nested trace + cost at http://localhost:3000

In [ ]:
from policy_copilot.tracing import load_traces

tr = pd.DataFrame(load_traces())
print(len(tr), "traces")
cols = [c for c in ["mode", "question", "refused", "input_tokens", "output_tokens", "cost_usd", "error"] if c in tr.columns]
tr[cols].tail(8)

## 6. Free play

Ideas:
- Raise `N` to 40 and run all answerable cases for the full failure distribution.
- For `missing_expected` cases, check whether the gold number is a **derived** value (not stated verbatim) — that's a scope decision.
- Turn any new failure you find into a golden case (diagnose → new golden).
- Compare `answer` vs `answer_agentic` on the same question by `cost_usd` / `n_search`.

In [ ]:
# Your scratch area
q = "What were Boeing's total revenues in 2022?"
r = answer_agentic(q, index, chunks)
print(r.text[:300])
print("verified:", r.verdict, "| searches:", len(r.tool_calls), "| cost $", r.cost_usd)